# Lab - System Prompts and Behaviour

Three tasks on the same customer complaint. You will see what the assistant does with
no rules, then give it rules, then make sure the rules survive a second turn.

| Task | What you learn |
| --- | --- |
| 1 | With no rules, the assistant promises whatever sounds nice |
| 2 | `system=` changes what it is willing to promise |
| 3 | The rules have to be sent on every single request |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

One thing worth knowing before you start: a request has two separate places for text.
The `messages` list is **what has been said**. The `system` argument is **how the
assistant should behave**. They are different arguments, and this lab is about the
second one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

# The two things the customer says, in order.
COMPLAINT = "Hi, I bought headphones last week and they do not work. I want my money back."
ORDER_NUMBER_REPLY = "The order number is ORD-12345678."

---

## Task 1 - Send the complaint with no system prompt

In [ ]:
# ============================================================
# TASK 1 - Send the complaint with no system prompt
# ============================================================
#
# WHAT TO DO
#   Send the customer's complaint the plain way, with no rules
#   attached, and read the reply carefully.
#
# WHY IT MATTERS
#   This is the "before" picture. An angry customer asks for
#   money back, and nothing in the request tells the assistant
#   what it is allowed to promise. Watch what it decides on its
#   own.
#
#   Read the reply and ask yourself two questions: did it
#   promise a refund, and did it ever ask which order this is?
#
# WHERE TO SEE IT IN THE LECTURE
#   "The messages list tells Claude what has been said so far.
#   The system prompt tells Claude how the assistant should
#   behave" - about 28 seconds in.
#
# HOW TO DO IT
#   One blank. Send COMPLAINT as a single user message.
#   Do NOT add a system argument here - that is the next task.
# ============================================================

# TODO: replace the ... below with the value named beside it
without_system = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {"role": "user", "content": ...},   # the COMPLAINT variable
    ],
)

print("Customer:   ", COMPLAINT)
print()
print("ShopAssist: ", without_system.content[0].text)

check("no_system", without_system=without_system)

---

## Task 2 - Send the same complaint with a system prompt

In [ ]:
# ============================================================
# TASK 2 - Send the same complaint with a system prompt
# ============================================================
#
# WHAT TO DO
#   Send the exact same complaint again, but this time attach
#   the rules below with the system argument. Then compare the
#   two replies.
#
# WHY IT MATTERS
#   The system prompt is not another message in the list. It is
#   its own argument, and it says who the assistant is, what
#   tone to use, and what it must not do. Read system_prompt
#   below - the line that changes everything here is "Do not
#   promise a refund until the order is checked."
#
#   Same customer, same words, different behaviour. Nothing
#   about the model changed. Only the request did.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now we can update our chat function to include the system
#   prompt" - about 1 minute 36 seconds in.
#
# HOW TO DO IT
#   Two blanks: the rules to attach, and the customer message
#   to send. Both are variables that already exist:
#
#       system_prompt   the rules, defined just below
#       COMPLAINT       the customer's words, from the setup cell
# ============================================================

system_prompt = """
You are ShopAssist AI, a helpful customer support assistant for an online store.

Your job is to help customers with order questions, returns, refunds, shipping issues,
and product support.

Be concise, polite, and practical.

Do not promise a refund until the order is checked.

If you need more information, ask one clear question at a time.
"""

# TODO: replace each ... below with the value named beside it
with_system = client.messages.create(
    model=model,
    max_tokens=300,
    system=...,   # the system_prompt variable defined just above
    messages=[
        {"role": "user", "content": ...},   # the COMPLAINT variable, unchanged
    ],
)

print("WITHOUT system prompt:")
print(without_system.content[0].text)
print()
print("WITH system prompt:")
print(with_system.content[0].text)

check("system_prompt", without_system=without_system, with_system=with_system)

---

## Task 3 - Keep the persona across turns

In [ ]:
# ============================================================
# TASK 3 - Keep the persona across turns
# ============================================================
#
# WHAT TO DO
#   Put the system prompt inside chat(), so it goes out on every
#   request, then run a two-turn conversation through it.
#
# WHY IT MATTERS
#   The system prompt is not remembered between calls, exactly
#   like the messages are not. If you attach it once and forget
#   it on the next call, the assistant quietly goes back to
#   promising refunds halfway through the conversation.
#
#   That is why it belongs inside chat(). Put it in one place
#   and it cannot be forgotten.
#
#   Watch both assistant turns below. Neither one should promise
#   a refund - not even after the customer supplies the order
#   number.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now let's test this again from the beginning" - about
#   2 minutes 24 seconds in.
#
# HOW TO DO IT
#   One blank, inside chat(). It is the same system_prompt
#   variable from Task 2 - the difference is that here it is
#   attached automatically, on every call.
# ============================================================

def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


# TODO: replace the ... below with the value named beside it
def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=300,
        system=...,   # the system_prompt variable - now on EVERY request
        messages=messages,
    )
    return message.content[0].text


# The rest of this cell is provided - it is the same two-turn pattern
# you built in the previous lab.
messages = []

add_user_message(messages, COMPLAINT)
add_assistant_message(messages, chat(messages))

add_user_message(messages, ORDER_NUMBER_REPLY)
add_assistant_message(messages, chat(messages))

for turn in messages:
    print("{:>9} | {}".format(turn["role"], turn["content"][:70].replace("\n", " ")))

check("persona_holds", messages=messages)

---

## Done

Same complaint, three requests, three different behaviours - and the model was the
same every time. What changed was what your application put in the request.

One architectural note before you move on. A system prompt is **guidance**, not
enforcement. "Do not promise a refund until the order is checked" is a good
instruction, and the model will usually follow it. But *usually* is not a guarantee,
and this is a rule about money. In Section 4 that rule moves into the backend tools,
where the model cannot talk its way around it.

Prompt is guidance. Application code is enforcement. That distinction runs through the
rest of this course.